In [1]:
import os
import cv2
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import OrderedDict
from PIL import Image
from mtcnn.mtcnn import MTCNN
import matplotlib.patches as patches
import keras
from sklearn.model_selection import train_test_split
import shutil
from shutil import unpack_archive
from subprocess import check_output

2023-11-29 18:34:20.945302: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Exploratory Data Analysis

In [2]:
# Set Data Path
DATA_PATH = '../data/lfw-dataset/lfw-deepfunneled/lfw-deepfunneled'


### Given Meta Data Information

In [3]:
# Get meta data
lfw_allnames = pd.read_csv("../data/lfw-dataset/lfw_allnames.csv")
matchpairsDevTest = pd.read_csv("../data/lfw-dataset/matchpairsDevTest.csv")
matchpairsDevTrain = pd.read_csv("../data/lfw-dataset/matchpairsDevTrain.csv")
mismatchpairsDevTest = pd.read_csv("../data/lfw-dataset/mismatchpairsDevTest.csv")
mismatchpairsDevTrain = pd.read_csv("../data/lfw-dataset/mismatchpairsDevTrain.csv")


- pairs.csv: Contains randomly generated splits for 10-fold cross validation specifically for pairs. Use this for the image restricted configuration when forming training sets (refer to readme). There are 10 total sets; 5 sets contain 300 matched pairs, the other 5 sets contain 300 mismatched pairs.

- people.csv: Contains randomly generated splits for 10-fold cross validation specifically for individual faces. Use this for the unrestricted configuration when forming training sets (refer to readme). There are 10 total sets, each with a different amount of people; Set 1: 601. Set 2: 555. Set 3: 552. Set 4: 560. Set 5: 567. Set 6: 527. Set 7: 597. Set 8: 601. Set 9: 580. Set 10: 609.

- matchpairsDevTest.csv: Use this testing set if you decide to go with the pairs configuration. Contains 500 matched pairs of faces for testing set.

- matchpairsDevTrain.csv: Use this training set if you decide to go with the pairs configuration. Contains 1100 matched pairs of faces for training set.

- mismatchpairsDevTest.csv: Use this testing set if you decide to go with the pairs configuration. Contains 500 mismatched pairs of faces for testing set.

- mismatchpairsDevTrain.csv: Use this training set f you decide to go with the pairs configuration. Contains 1100 mismatched pairs of faces for training set.

- peopleDevTest.csv: Use this testing test if you decide to go with the people configuration. Contains 1711 people and 3708 images.

- peopleDevTrain.csv: Use this training set if you decide to go with the people configuration. Contains 4038 people and 9525 images.

In [4]:
pairs = pd.read_csv("../data/lfw-dataset/pairs.csv")
# tidy pairs data: 
pairs = pairs.rename(columns ={'name': 'name1', 'Unnamed: 3': 'name2'})
matched_pairs = pairs[pairs["name2"].isnull()].drop("name2",axis=1)
mismatched_pairs = pairs[pairs["name2"].notnull()]
people = pd.read_csv("../data/lfw-dataset/people.csv")
# remove null values
people = people[people.name.notnull()]
peopleDevTest = pd.read_csv("../data/lfw-dataset/peopleDevTest.csv")

- **lfwallnames.csv:** Contains all names of each face in the dataset along with number of images each face has.

- **lfwreadme.csv:** Comprehensive readme file found on the original database. If there is any information you are missing here or are looking for additional resources you will probably find it in this file. It explains how each .csv file comes into play when forming training and testing models, as well as column metadata information for figuring out what the .csv is talking about. The original website also gives recommendations on training/testing splits and comparison benchmarks.


In [5]:
print("Number of unique celbrities: ",len(lfw_allnames))

Number of unique celbrities:  5749


In [6]:
print("Celebrities with multiple images: ", sum(lfw_allnames.images > 1))

Celebrities with multiple images:  1680


In [7]:
print(" Most photographed celebrities: ", lfw_allnames.sort_values(by="images",ascending=False).head(10))

 Most photographed celebrities:                     name  images
1871      George_W_Bush     530
1047       Colin_Powell     236
5458         Tony_Blair     144
1404    Donald_Rumsfeld     121
1892  Gerhard_Schroeder     109
373        Ariel_Sharon      77
2175        Hugo_Chavez      71
2941  Junichiro_Koizumi      60
2468      Jean_Chretien      55
2682      John_Ashcroft      53


In [13]:
def load_image(name, number, data_path=DATA_PATH):
    """
    Load an image from the dataset.
    :param name: Name of the person.
    :param number: Image number for the person.
    :param data_path: Base directory of the LFW dataset.
    :return: The loaded image.
    """
    filename = f"{name}_{number:04d}.jpg"
    filepath = os.path.join(data_path, name, filename)
    image = cv2.imread(filepath)
    return image


In [14]:
def preprocess_image(image, target_size=(224, 224)):
    """
    Preprocess the image by resizing and normalizing.
    :param image: The image to preprocess.
    :param target_size: The target size of the image.
    :return: Preprocessed image.
    """
    image = cv2.resize(image, target_size)
    image = image / 255.0  # Normalize pixel values
    return image

In [15]:
def create_dataset(data_path, allnames, target_size=(224, 224)):
    """
    Create a dataset of images and labels.
    :param data_path: Base directory of the LFW dataset.
    :param allnames: DataFrame containing names and number of images.
    :param target_size: Target size for each image.
    :return: List of preprocessed images and labels.
    """
    images = []
    labels = []

    for index, row in allnames.iterrows():
        name = row['name']
        for i in range(1, row['images'] + 1):
            image = load_image(name, i, data_path)
            image = preprocess_image(image, target_size)
            images.append(image)
            labels.append(name)

    return np.array(images), np.array(labels)

# Create the dataset
images, labels = create_dataset(DATA_PATH, lfw_allnames)
